# Workshop 04 — Python, NumPy & Linear Regression

We first learn one-feature linear regression from horizontal motion measurements. Then we switch to a synthetic house-price dataset to practise two-feature regression.

The [interactive lesson](https://siadu.github.io/MLMPworkshop/workshop04/) explains the data, Python basics, and the mathematics. **This Notebook is where you write and test the code.** Run cells from top to bottom.

| Section | What you will do |
|---|---|
| 00 · Motion data | Read the measurements and plot them |
| 01 · Python foundations | Write functions using lists, loops and conditions |
| 02 · NumPy & shapes | Work with arrays and matrix operations |
| 03 · Prediction & loss | Implement a linear prediction and MSE |
| 04 · Gradient descent | Calculate gradients and train a model |
| 05 · Multiple features | Use house size and house age as inputs |
| 06 · Train & compare | Compare one-feature and two-feature house-price models |

**Naming throughout:** `x` = time in seconds; `y` = measured horizontal position; `y_pred` = predicted position; `w` = weight; `b` = bias; `N` = number of samples; `error = y_pred - y`; `loss` = mean squared error (MSE); `X` = input matrix with more than one feature.


## Setup

Download this `.ipynb` file and place it in `MLMPworkshop/workshop04_Python_NumPy_LinearRegression/`. The project should be in `~/Desktop/MLMPworkshop`.

```bash
cd ~/Desktop/MLMPworkshop/workshop04_Python_NumPy_LinearRegression/
ls
# Workshop04_Python_NumPy_LinearRegression.ipynb
uv init
uv venv --python 3.11
source .venv/bin/activate  # Linux / macOS
uv add numpy matplotlib ipykernel notebook
```

Go to `.gitignore` and add `.ipynb_checkpoints` as the last line.


## 00 · Motion Data

The character moves from left to right. We record its horizontal position at 11 frames.

- `frames`: the frame number (0 to 30, measured every 3 frames).
- `positions`: the measured horizontal position, in scene units.
- `fps = 30`: the animation plays 30 frames per second.
- One `(time, position)` pair is **one sample**.

We use time in **seconds** as the input: `x = frame / fps`. At 30 fps, frame 15 is at 0.5 seconds. **Converting frames to seconds is not required by linear regression.** It gives us a useful time unit and keeps the input values small for gradient descent.

**Input:** one feature (time). **Target:** one horizontal position per sample. The interactive lesson includes a sharp, frame-by-frame SVG motion animation and the full data table.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

frames = [0, 3, 6, 9, 12, 15, 18, 21, 24, 27, 30]
positions = [1.52, 2.02, 2.52, 3.01, 3.51, 4.01, 4.51, 5.01, 5.5, 6.0, 6.5]
fps = 30
assert len(frames) == len(positions)

def plot_data(x, y, title="Observed motion"):
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.scatter(x, y, color="#1f78b4", s=45, label="Measured position")
    ax.set(xlabel="Time x (seconds)", ylabel="Horizontal position y", title=title)
    ax.grid(alpha=0.25)
    ax.legend()
    plt.show()

plot_data(np.array(frames) / fps, np.array(positions))
print("Samples:", len(frames))
print("First sample:", frames[0], "→", positions[0])

**Next:** Return to the interactive lesson, **Section 01 — Python Foundations**.

## 01 · Python Foundations

Use the examples from the interactive lesson as a reference for valid variable names, `list`, index, `len()`, `print()`, `for`, `if` / `elif` / `else`, `def` and `return`.

In the next two tasks, use a **loop** to calculate a result and then a **function** to reuse it. Your functions must return the answer, not just print it.


In [ ]:
values = [3, 7, 2]
print("First value:", values[0])        # Index starts at 0
print("Number of values:", len(values))
total = 0
for value in values:
    total += value                       # Same as: total = total + value
print("Total:", total)

### Task 1 — Write `manual_mean()`

Return the arithmetic mean of a list **without** using `sum()` or NumPy. Return `None` for an empty list. Reuse the loop pattern above.

In [ ]:
def manual_mean(values):
    # TODO: Use a for loop to calculate the mean. Return None for an empty list.
    raise NotImplementedError("Task 1")


In [ ]:
assert manual_mean([]) is None
assert manual_mean([2, 4, 6]) == 4
assert manual_mean([-2, 0, 2]) == 0
print("Task 1 passed")

### Task 2 — Reuse a function

`count_above_mean(values)` returns how many values are **strictly greater** than their mean. Example: `[3, 7, 2, 9, 4]` has mean `5`; return `2`. Return `0` for an empty list. Call `manual_mean()` instead of calculating the mean again.

In [ ]:
def count_above_mean(values):
    # TODO: Count values strictly greater than manual_mean(values).
    raise NotImplementedError("Task 2")


In [ ]:
assert count_above_mean([3, 7, 2, 9, 4]) == 2
assert count_above_mean([2, 2, 2]) == 0
assert count_above_mean([-2, -1, -3]) == 1
assert count_above_mean([]) == 0
print("Task 2 passed")

**Next:** Return to the interactive lesson, **Section 02 — NumPy & Shapes**.

## 02 · NumPy & Shapes

NumPy arrays store numerical data and support matrix operations. The interactive lesson shows matrix multiplication first, then broadcasting, then samples and features.

- `(N,)`: a 1D array with `N` values.
- `(N, 1)`: a 2D array with `N` rows and one feature column.
- `(N, 2)`: a 2D array with `N` rows and two feature columns.

`.reshape(-1, 1)` creates a column array; `-1` asks NumPy to work out the row count. `A[:, 0]` selects the first column.


In [ ]:
example = np.array([10., 20., 30.])
example_column = example.reshape(-1, 1)
print("1D:", example.shape)
print("Column:", example_column.shape)
print("First column:", example_column[:, 0])

### Task 3 — Turn animation data into arrays

Create `x` from frames expressed in **seconds** and `y` from the measured positions. Both are **1D arrays**. Then use `.reshape(-1, 1)` to build `x_column`.

**Useful syntax:** `np.array(values, dtype=float)`, `/ fps`, `.reshape(-1, 1)`.


In [ ]:
# TODO: Create x (seconds), y (measured positions) and x_column (N, 1).
raise NotImplementedError("Task 3")


In [ ]:
N = len(frames)
assert x.shape == (N,) and y.shape == (N,)
assert x_column.shape == (N, 1)
assert np.allclose(x, np.arange(0, 31, 3) / fps)
assert np.allclose(x_column[:, 0], x)
print("Task 3 passed:", "x", x.shape, "x_column", x_column.shape)

### Multiplication: `*` vs `@`

- `A * w`: element-wise multiplication. NumPy may broadcast a vector across rows.
- `A @ w`: matrix multiplication. Each row combines with the weight vector to produce one result.
- `A @ B`: matrix multiplication with two matrices, producing a new matrix.

Check the output shapes as well as the numerical values.


In [ ]:
A_demo = np.array([[1., 2.], [3., 4.], [5., 6.]])  # 3 rows, 2 columns
w_demo = np.array([0.2, 0.8])
print("A * w:\n", A_demo * w_demo)
print("A @ w:", A_demo @ w_demo)
print("Output shape:", (A_demo @ w_demo).shape)

B_demo = np.array([[1., 2.], [0., 1.], [2., 0.]])
C_demo = np.array([[1., 2., 3.], [4., 5., 6.]])
print("Matrix @ matrix:\n", C_demo @ B_demo)
print("Matrix output shape:", (C_demo @ B_demo).shape)


### Task 4 — Predict shapes, then calculate

Write a function that returns **one number per row** of a 2D array. Use `@` rather than a loop or `*`.

In [ ]:
def matrix_product(X, w):
    # TODO: Return the matrix multiplication result.
    raise NotImplementedError("Task 4")


In [ ]:
assert matrix_product(A_demo, w_demo).shape == (3,)
assert np.allclose(matrix_product(A_demo, w_demo), [1.8, 3.8, 5.8])
print("Task 4 passed")

**Next:** Return to the interactive lesson, **Section 03 — Prediction & Loss**.

## 03 · Prediction & Loss

With one input feature, the prediction is **ŷ = wx + b**.

- `x`: time in seconds; `w`: weight (slope); `b`: bias (value at `x=0`).
- `y`: measured position; `y_pred` or `ŷ`: predicted position.
- `error = y_pred - y`: the residual for each sample.
- **MSE** (`loss`) = the mean of squared residuals. Smaller MSE means a closer fit **on these samples**.

$$
\mathrm{MSE}=\frac{1}{N}\sum_{i=1}^{N}(\hat{y}_i-y_i)^2
$$

`N` is the number of samples, `i` selects one sample, `ŷᵢ` is its predicted position, `yᵢ` is its measured position, and `Σ` means to sum across all samples.


### Task 5 — Prediction

Complete `predict(x, w, b)`. It should work for one number or a whole NumPy array.

In [ ]:
def predict(x, w, b):
    # TODO: Implement one-feature linear regression.
    raise NotImplementedError("Task 5")


In [ ]:
assert np.isclose(predict(2., 3., 1.), 7.)
assert np.allclose(predict(np.array([0., 1.]), 2., 1.), [1., 3.])
print("Task 5 passed")

### Task 6 — MSE: Python first, NumPy next

Given two equal-length arrays, return `mean((y_pred - y)²)`.

**Part A:** use a Python `for` loop and a running total. **Part B:** write the same result with `np.mean()` and array operations. Use the NumPy version later in training.

In [ ]:
def mse_loop(y, y_pred):
    # TODO: Compute the mean squared error with a Python loop.
    raise NotImplementedError("Task 6A")

def compute_mse(y, y_pred):
    # TODO: Compute the mean squared error with NumPy.
    raise NotImplementedError("Task 6B")


In [ ]:
test_y = np.array([1., 2., 4.])
test_pred = np.array([2., 1., 5.])
assert np.isclose(mse_loop(test_y, test_pred), 1.)
assert np.isclose(compute_mse(test_y, test_pred), 1.)
assert np.isclose(compute_mse(y, y), 0.)
print("Task 6 passed")

### See the prediction

Change `w_try` and `b_try`. Notice how the straight line and MSE change. The plotting code is provided so you can focus on the model.


In [ ]:
def plot_fit(x, y, y_pred, title="Prediction"):
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.scatter(x, y, label="Measured position", s=40, color="#2563eb")
    ax.plot(x, y_pred, label="Predicted position", color="#e68a21", linewidth=2)
    ax.set(xlabel="Time x (seconds)", ylabel="Horizontal position y", title=title)
    ax.grid(alpha=0.25)
    ax.legend()
    plt.show()

w_try, b_try = 0.0, 0.0  # Change these two values
y_try = predict(x, w_try, b_try)
print("MSE:", compute_mse(y, y_try))
plot_fit(x, y, y_try)

**Next:** Return to the interactive lesson, **Section 04 — Gradient Descent**.

## 04 · Gradient Descent

A **derivative** describes how one value changes when another changes. A **partial derivative** describes the change with respect to one parameter while the others stay fixed.

Our `loss` is MSE. For one sample: `y_pred = w*x+b`, `error = y_pred-y`, and squared error `L = error**2`. The interactive lesson explains the chain rule: changing `w` changes `y_pred`, which changes `error`, which changes `L`.

- Single-sample gradients: `∂L/∂w = 2*error*x` and `∂L/∂b = 2*error`.
- MSE averages these contributions over `N` samples, giving the factor **2/N**.

$$
dw=\frac{\partial\mathrm{loss}}{\partial w}=\frac{2}{N}\sum_{i=1}^{N}error_i x_i
$$

$$
db=\frac{\partial\mathrm{loss}}{\partial b}=\frac{2}{N}\sum_{i=1}^{N}error_i
$$

**Why does the main course use 1/(2N)?** Its alternative cost `J = MSE/2` cancels the factor 2 when differentiated. Our code uses `loss = MSE`, so the gradients above keep the 2. Both costs have the same minimum.

**NumPy syntax:** `dw = 2*np.mean(error * x)`; `db = 2*np.mean(error)`.


### Task 7 — Gradients

Complete `compute_gradients()`. Return two **scalar numbers**, `dw` and `db`. Reuse `predict()`.

In [ ]:
def compute_gradients(x, y, w, b):
    # TODO: Return the gradients dw and db for MSE.
    raise NotImplementedError("Task 7")


In [ ]:
dw_test, db_test = compute_gradients(np.array([0., 1.]), np.array([1., 3.]), 0., 0.)
assert np.isclose(dw_test, -3.) and np.isclose(db_test, -4.)
print("Task 7 passed")

### Task 8 — Update the parameters

Gradient descent moves **against the gradient**:

**new w = old w − learning_rate × dw**  
**new b = old b − learning_rate × db**

The **learning rate** `alpha` controls step size. Calculate both gradients *before* changing either parameter.

In [ ]:
def gradient_step(w, b, dw, db, alpha):
    # TODO: Update both parameters and return them.
    raise NotImplementedError("Task 8")


In [ ]:
w_check, b_check = gradient_step(0., 0., -3., -4., 0.1)
assert np.isclose(w_check, 0.3) and np.isclose(b_check, 0.4)
print("Task 8 passed")

### Task 9 — Training loop

Each **epoch**: **predict → calculate loss → calculate gradients → update parameters**. Store `loss` so we can plot whether training is working.

In [ ]:
def train(x, y, alpha=0.1, epochs=400):
    # TODO: Reuse predict(), compute_mse(), compute_gradients(), gradient_step().
    # Store one loss value per epoch. Return w, b, losses.
    raise NotImplementedError("Task 9")


In [ ]:
def plot_losses(losses, title="Loss during training"):
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(losses, color="#0d9488")
    ax.set(xlabel="Epoch", ylabel="MSE", title=title)
    ax.grid(alpha=0.25)
    plt.show()

w_fit, b_fit, losses = train(x, y, alpha=0.1, epochs=400)
assert len(losses) == 400
assert np.isfinite(losses).all()
assert losses[-1] < losses[0] * 0.05
print("Learned w:", round(w_fit, 4), "Learned b:", round(b_fit, 4))
print("MSE before / after:", round(losses[0], 4), round(losses[-1], 4))
plot_fit(x, y, predict(x, w_fit, b_fit), "Trained model")
plot_losses(losses)

### Experiment — Learning rate

Try several values of `alpha`. A very small value may need more epochs; a very large value can make the loss grow. The data use seconds (0–1), rather than raw frame numbers (0–30), so the same update has a more manageable numerical scale.


In [ ]:
for alpha_try in [0.001, 0.1, 1.5]:
    w_result, b_result, loss_result = train(x, y, alpha=alpha_try, epochs=400)
    print("alpha:", alpha_try, "last MSE:", round(loss_result[-1], 5))

**Next:** Return to the interactive lesson, **Section 05 — Multiple Features**.

## 05 · Multiple Features

We now switch to a **house-price dataset**. Each house has two input features:

- `size_m2`: house size in square metres.
- `age_years`: house age in years.

The target is `price_100k`: house price, measured in units of **£100,000**. For example, `2.50` means **£250,000**.

We predict **one price** with two input features:

$$
\hat{y}=w_1x_1+w_2x_2+b
$$

The two features have different units and numerical ranges, so we scale them before training:

- `size_x = size_m2 / 100` (size measured in hundreds of m²).
- `age_x = age_years / 50` (age measured in groups of 50 years).

**Scaling does not add or remove any features.** Our input matrix is `X_house = np.column_stack([size_x, age_x])` with shape `(N, 2)`. We keep one price target: `price_100k` with shape `(N,)`.

This is a small, **synthetic teaching dataset**, not actual UK property prices. The linear relationship is deliberately simple for practising regression.


In [ ]:
# Eleven example houses (synthetic data, not market observations).
size_m2 = np.array([55, 75, 90, 110, 125, 145, 160, 65, 100, 135, 175], dtype=float)
age_years = np.array([5, 30, 12, 40, 8, 25, 45, 42, 3, 35, 15], dtype=float)
price_100k = np.array([2.410, 2.370, 2.928, 2.840, 3.622, 3.710, 3.660, 1.998, 3.252, 3.370, 4.410])

# Divide by reference values to give the two features a similar scale.
size_x = size_m2 / 100.0
age_x = age_years / 50.0
X_house = np.column_stack([size_x, age_x])

print("House size (m²):", size_m2)
print("House age (years):", age_years)
print("Price (£100k):", price_100k)
print("X_house shape:", X_house.shape, "target shape:", price_100k.shape)
print("First three feature rows:\n", X_house[:3])


### Task 10 — Predict one output from two inputs

Implement `predict_multi(X, w, b)` using `X @ w + b`. One house is one row; the output is **one predicted price per house**.


In [ ]:
def predict_multi(X, w, b):
    # TODO: Return a vector of predicted prices.
    raise NotImplementedError("Task 10")

In [ ]:
result = predict_multi(X_house, np.array([2., -1.]), 1.)
assert result.shape == price_100k.shape
assert np.allclose(predict_multi(np.array([[1., 2.], [3., 4.]]),
                                 np.array([2., 1.]), 0.5), [4.5, 10.5])
print("Task 10 passed")

### Matrix gradient — prepare for Section 06

Each feature has one weight and one gradient. The gradient vector `dw` contains both values.

$$
d\mathbf{w}=\frac{2}{N}X^T\mathbf{error},\qquad db=2\operatorname{mean}(\mathbf{error})
$$

| Math | NumPy syntax | Shape |
|---|---|---|
| House input matrix | `X_house` | `(N, 2)` |
| Transposed input | `X_house.T` | `(2, N)` |
| Residuals | `error` | `(N,)` |
| Weight gradients | `(2/N) * (X_house.T @ error)` | `(2,)` |

`.T` swaps the rows and columns so the residuals can be combined separately with each feature. The bias gradient is a single scalar.


In [ ]:
print("X_house shape:", X_house.shape)
print("X_house.T shape:", X_house.T.shape)
error_demo = np.array([1., -1., 2.])
print("First 3 feature rows transposed:", X_house[:3].T.shape)
print("Two feature sums:", X_house[:3].T @ error_demo)

**Next:** Return to the interactive lesson, **Section 06 — Train & Compare**.

## 06 · Train & Compare

Can we predict the **same house price** more accurately by using both size and age?

**Step 1:** Train a one-feature model with `size_x` → `price_100k`.

**Step 2:** Train a two-feature model with `X_house = [size_x, age_x]` → `price_100k`.

**Step 3:** Compare predictions and MSE. Both models predict **one price**, not two outputs.


In [ ]:
# Reuse the house data defined in Section 05.
assert X_house.shape == (11, 2) and price_100k.shape == (11,)
print("One-feature input:", size_x.shape)
print("Two-feature input:", X_house.shape)
print("Target (price in £100k):", price_100k.shape)

### Step 1 — Train a one-feature model

Reuse `train()` from Section 04 to predict `price_100k` using **only** `size_x`. Store `w_linear`, `b_linear`, and `linear_losses`.


In [ ]:
# TODO: Fit the one-feature model with size_x and price_100k.
# Save w_linear, b_linear and linear_losses.
raise NotImplementedError("Step 1")

### Step 2 — Train a two-feature model

Write `train_multi(X, y, ...)` following the training loop from Section 04, but use **a vector of two weights** instead of one scalar weight.

Start with `w = np.zeros(X.shape[1])`, `b = 0.0` and an empty list `losses`. For each epoch: predict → residuals → MSE → `dw = (2/N) * (X.T @ error)` and `db = 2 * np.mean(error)` → update both weights and the bias. Return `(w, b, losses)`.


In [ ]:
def train_multi(X, y, alpha=0.1, epochs=1500):
    # TODO: Initialise a weight vector, bias, and a list for MSE.
    # TODO: Loop over the samples and update both weights together.
    # TODO: Return w, b, losses.
    raise NotImplementedError("Step 2")

### Step 3 — Compare predictions

Use the provided plotting helper to show actual prices and the two sets of predictions. The horizontal axis shows **house size in m²**; the vertical axis shows **house price in £100,000 units**.

The MSE graph compares how the two models learn on the same dataset.


In [ ]:
def plot_comparison(size_m2, y_true, one_pred, two_pred, one_losses, two_losses):
    order = np.argsort(size_m2)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    axes[0].scatter(size_m2, y_true, label="Observed price", color="#2563eb")
    axes[0].plot(size_m2[order], one_pred[order], label="Size only", color="#e68a21")
    # A two-feature model does not form a single price-vs-size line when age varies.
    axes[0].scatter(size_m2, two_pred, label="Size + age", color="#0d9488", marker="x", s=65)
    axes[0].set(xlabel="House size (m²)", ylabel="Price (£100k)", title="Predicted house prices")
    axes[0].legend()
    axes[1].plot(one_losses, label="Size only", color="#e68a21")
    axes[1].plot(two_losses, label="Size + age", color="#0d9488")
    axes[1].set(xlabel="Epoch", ylabel="MSE", title="Training loss")
    axes[1].legend()
    for ax in axes: ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

# TODO: Fit train_multi(X_house, price_100k, ...) to get w_multi, b_multi, multi_losses.
# TODO: Calculate one_pred and two_pred with your trained parameters.
# TODO: Call plot_comparison(size_m2, price_100k, ...).
raise NotImplementedError("Step 3")

### Check your results

- Which model predicts the house prices more accurately on these samples?
- What information can age add when two houses have similar sizes?
- What are the two values in `w_multi`?
- Does a low training MSE guarantee that the model predicts prices of other houses accurately?

**Your observations:**

> Write here.
